# SOM v4 node-weighted для прогноза перспективности

Версия v4 усиливает именно **узлы наложения геологических факторов**. В отличие от v3, узловые признаки влияют не только на жёлтые `top_zone`, но и на само обучение SOM и на оценку SOM-нейронов.

Главная идея: одиночные разломы не должны сами по себе делать зону перспективной; выше оцениваются участки, где совпадают тектоника, магматизм, палеодолины, структурные объекты и фации.

In [ ]:
# # SOM v4: node-weighted Self-Organizing Map для прогноза перспективности
#
# Этот ноутбук делает прогноз **ML-методом без учителя** — через **самоорганизующуюся карту Кохонена (SOM)**.
#
# Версия v3 отличается от v2:
#
# 1. **Уменьшено дублирование тектонических признаков**, чтобы результат не превращался просто в карту разломов.
# 2. Из буферных признаков по умолчанию оставлены только `1000` и `1500 м`; локальный буфер `500 м` убран.
# 3. Тектоника в бинарных признаках учитывается как один класс `tect_any`, а не как несколько повторяющихся слоёв.
# 4. `top_zone` сделана строже: повышен квантиль, увеличен размер локального максимума и минимальный размер компоненты.
# 5. Добавлены диагностические таблицы: какие SOM-нейроны стали перспективными и какие признаки у них выше среднего.
#
# Главная идея:
#
# - SOM обучается **на всех ячейках территории**, без меток `0/1`.
# - Известные точки и геохимические признаки **не обучают SOM**, а используются только после обучения для интерпретации кластеров.
# - Итог — `prospectivity` и обратный слой `prognoz = 1 - prospectivity`.



# Если каких-то библиотек нет, установите их вручную в отдельной ячейке:
# !pip install geopandas shapely pyproj scikit-learn scipy matplotlib pandas numpy

from pathlib import Path
import json
import math
import random
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep
from pyproj import CRS

from sklearn.preprocessing import RobustScaler
from scipy.ndimage import gaussian_filter, maximum_filter, label

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)



# =========================
# CONFIG
# =========================
BASE_DIR = Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз")
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "som_unsupervised_ml_v4_nodeweighted_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CELL_SIZE = 500

LAYER_MAP = {
    "mask": "svita_new",
    "facies": "fasii",
    "paleo": "gr_dol_vp_poly",
    "struct": "kory",
    "magm": "dayki_buf",
    "tect1": "glub_raz_nw",
    "tect2": "glub_r_nw",
}

# Evidence используется НЕ для обучения SOM, а только для интерпретации SOM-кластеров.
EVIDENCE_POINT_LAYER_NAMES = ["result", "геохимическое_опробование"]

# В v2 по умолчанию polygon-ореолы выключены, чтобы SOM не становилась слишком привязанной
# к крупным геохимическим площадям. Если нужно учитывать их как evidence, поставьте True.
USE_EVIDENCE_POLYGONS = False
EVIDENCE_POLYGON_LAYER_NAMES = ["геохимические ореолы", "привнос урана"]

# SOM параметры. Если долго — уменьшите SOM_EPOCHS или SOM_MAX_SAMPLES_PER_EPOCH.
SOM_X = 12
SOM_Y = 12
SOM_EPOCHS = 45
SOM_MAX_SAMPLES_PER_EPOCH = 5000
SOM_START_LR = 0.40
SOM_END_LR = 0.04
SOM_START_SIGMA = max(SOM_X, SOM_Y) / 2.0
SOM_END_SIGMA = 0.9
SOM_PRINT_EVERY = 5

# Как сильно сглаживать evidence по SOM-сетке.
SOM_SCORE_SMOOTH_SIGMA = 1.0
EVIDENCE_PRIOR_STRENGTH = 8.0

# В v4 главная цель — изменить не только top-zone, но и сам фон карты.
# Поэтому узловые признаки усиливаются на входе SOM и при интерпретации SOM-нейронов.
INTRA_NEURON_WEIGHT = 0.04
NODE_SCORE_WEIGHT = 0.28

# Веса для итогового score SOM-нейрона:
# evidence — где известные проявления чаще попадают в SOM-нейрон;
# node — насколько нейрон связан с узлами пересечения факторов;
# active — сколько разнородных факторов в среднем совпадает в ячейках нейрона.
NEURON_EVIDENCE_WEIGHT = 0.45
NEURON_NODE_WEIGHT = 0.35
NEURON_ACTIVE_WEIGHT = 0.20

# Групповые веса признаков перед обучением SOM.
# Одиночная тектоника ослабляется, узловые сочетания усиливаются.
TECT_FEATURE_MULTIPLIER = 0.65
FACIES_FEATURE_MULTIPLIER = 0.95
MAGM_FEATURE_MULTIPLIER = 1.05
PALEO_FEATURE_MULTIPLIER = 1.15
STRUCT_FEATURE_MULTIPLIER = 1.20
NODE_FEATURE_MULTIPLIER = 2.10
FEATURE_CLIP_MAX = 2.50

# Пространственное сглаживание итоговой карты.
SMOOTH_PASSES = 2
TOP_ZONE_Q = 0.993
LOCAL_MAX_SIZE = 5
MIN_TOP_COMPONENT_CELLS = 3
MAX_TOP_ZONE_SHARE = 0.008
NODE_TOP_ZONE_Q = 0.70

# Масштабы для proximity-признаков.
SCALES = {
    "facies": 1200.0,
    "paleo": 1200.0,
    "struct": 900.0,
    "magm": 900.0,
    "tect1": 1000.0,
    "tect2": 1000.0,
}

# В v2 убран 500 м: он слишком резко подсвечивает линии и усиливает разломы.
BUFFER_DISTANCES = [1000, 1500]

SEED = 42
SHOW_POINTS = True
SAVE_FULL_GRID_GPKG = True

random.seed(SEED)
np.random.seed(SEED)




## 1. Вспомогательные функции

In [ ]:
#
# Здесь специально используется мягкая загрузка слоёв:
# - читаются `*.shp`;
# - если CRS не найден, проверяются файлы вида `*_shp.pj4`, `.pj4`, `.qpj`, `.prj`;
# - геометрии не чинятся агрессивно через `buffer(0)`, чтобы не “съесть” маску.



def read_sidecar_proj4(path: Path):
    candidates = [
        path.with_name(path.stem + "_shp.pj4"),
        path.with_suffix(".pj4"),
        path.with_suffix(".qpj"),
        path.with_suffix(".prj"),
    ]
    for sidecar in candidates:
        if sidecar.exists():
            txt = sidecar.read_text(encoding="utf-8", errors="ignore").strip()
            if not txt:
                continue
            for line in txt.splitlines():
                line = line.strip()
                if line.lower().startswith("pj4="):
                    return line.split("=", 1)[1].strip()
            return txt
    return None


def repair_geometries(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    if "geometry" not in gdf:
        return gdf
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    return gdf


def finite_bounds(gdf: gpd.GeoDataFrame, layer_name: str = "layer"):
    if gdf is None or len(gdf) == 0:
        raise ValueError(f"Слой {layer_name} пуст после базовой очистки геометрий.")
    bounds = np.asarray(gdf.total_bounds, dtype=float)
    if len(bounds) != 4 or not np.all(np.isfinite(bounds)):
        raise ValueError(
            f"Некорректные total_bounds у слоя {layer_name}: {bounds}. "
            "Проверь CRS, исходную геометрию и sidecar-проекцию."
        )
    minx, miny, maxx, maxy = bounds.tolist()
    if maxx <= minx or maxy <= miny:
        raise ValueError(f"У слоя {layer_name} вырожденные границы: {(minx, miny, maxx, maxy)}")
    return minx, miny, maxx, maxy


def load_layer(path: Path) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)
    gdf = repair_geometries(gdf)
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(path)
        if proj4:
            try:
                gdf = gdf.set_crs(CRS.from_user_input(proj4), allow_override=True)
            except Exception as e:
                print(f"Не удалось прочитать CRS из {path.name}: {e}")
    return gdf


def load_shapefiles(folder: Path):
    layers = {}
    for shp in sorted(folder.glob("*.shp")):
        try:
            layers[shp.stem] = load_layer(shp)
        except Exception as e:
            print(f"Не удалось загрузить {shp.name}: {e}")
    return layers


def to_crs_safe(gdf: gpd.GeoDataFrame, target_crs):
    if target_crs is None:
        return gdf
    if gdf.crs is None:
        return gdf.set_crs(target_crs, allow_override=True)
    if str(gdf.crs) == str(target_crs):
        return gdf
    return gdf.to_crs(target_crs)


def choose_target_crs(mask_gdf: gpd.GeoDataFrame):
    crs = mask_gdf.crs
    if crs is None:
        return None
    try:
        if crs.is_projected:
            return crs
    except Exception:
        pass
    try:
        utm = mask_gdf.estimate_utm_crs()
        if utm is not None:
            return utm
    except Exception:
        pass
    return crs


def harmonize_layers(layers: dict, target_crs):
    out = {}
    for name, gdf in layers.items():
        try:
            cur = to_crs_safe(gdf, target_crs)
            cur = repair_geometries(cur)
            out[name] = cur
        except Exception as e:
            print(f"Не удалось привести CRS слоя {name}: {e}")
            out[name] = gdf
    return out


def discover_point_layers(layers: dict, exclude_names=None):
    exclude_names = set(exclude_names or [])
    candidates = []
    for name, gdf in layers.items():
        if name in exclude_names or len(gdf) == 0:
            continue
        geom_types = set(gdf.geom_type.dropna().unique().tolist())
        if geom_types.issubset({"Point", "MultiPoint"}):
            candidates.append({
                "layer": name,
                "n": int(len(gdf)),
                "geom_types": sorted(list(geom_types)),
            })
    return candidates


def normalize_01(x):
    x = np.asarray(x, dtype=float)
    mn = np.nanmin(x)
    mx = np.nanmax(x)
    if not np.isfinite(mn) or not np.isfinite(mx) or mx <= mn:
        return np.zeros_like(x, dtype=float)
    return (x - mn) / (mx - mn)


def robust_normalize_01(x, q_low=0.02, q_high=0.98):
    x = np.asarray(x, dtype=float)
    lo = np.nanquantile(x, q_low)
    hi = np.nanquantile(x, q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return normalize_01(x)
    y = (x - lo) / (hi - lo)
    return np.clip(y, 0.0, 1.0)


def distance_to_proximity(distances, scale):
    distances = np.asarray(distances, dtype=float)
    return 1.0 / (1.0 + distances / float(scale))


def build_grid(mask_gdf: gpd.GeoDataFrame, cell_size=500):
    minx, miny, maxx, maxy = finite_bounds(mask_gdf, "mask")
    mask_union = unary_union(mask_gdf.geometry)
    prepared_mask = prep(mask_union)

    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = len(ys)
    cols = len(xs)
    if rows <= 0 or cols <= 0:
        raise ValueError(f"Некорректный размер сетки: rows={rows}, cols={cols}")

    geoms = []
    rr = []
    cc = []
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            cell = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(cell):
                geoms.append(cell)
                rr.append(r)
                cc.append(c)

    if len(geoms) == 0:
        raise ValueError("После пересечения с маской сетка пустая. Проверь CRS слоёв и cell_size.")

    grid = gpd.GeoDataFrame({"row": rr, "col": cc}, geometry=geoms, crs=mask_gdf.crs)
    grid["cell_id"] = np.arange(len(grid))
    grid["centroid"] = grid.geometry.centroid
    return grid, (rows, cols), (minx, miny, maxx, maxy)


def compute_distance_feature(grid, source_gdf, scale):
    if source_gdf is None or len(source_gdf) == 0:
        dist = np.full(len(grid), np.nan, dtype=float)
        prox = np.zeros(len(grid), dtype=float)
        return dist, prox
    union_geom = unary_union(source_gdf.geometry)
    dist = grid["centroid"].distance(union_geom).to_numpy(dtype=float)
    prox = distance_to_proximity(dist, scale)
    return dist, prox


def compute_intersection_proximity(grid, gdf_a, gdf_b, scale=1200.0):
    """
    Признак близости к реальному геометрическому узлу пересечения двух слоёв.
    Работает для линий, полигонов и их границ. Если пересечения нет, возвращает нули.
    """
    try:
        if gdf_a is None or gdf_b is None or len(gdf_a) == 0 or len(gdf_b) == 0:
            return np.full(len(grid), np.nan, dtype=float), np.zeros(len(grid), dtype=float)
        geom_a = unary_union(gdf_a.geometry)
        geom_b = unary_union(gdf_b.geometry)
        inter = geom_a.intersection(geom_b)
        if inter is None or inter.is_empty:
            # Для полигональных слоёв иногда полезнее пересечение границ.
            try:
                inter = geom_a.boundary.intersection(geom_b.boundary)
            except Exception:
                pass
        if inter is None or inter.is_empty:
            return np.full(len(grid), np.nan, dtype=float), np.zeros(len(grid), dtype=float)
        dist = grid["centroid"].distance(inter).to_numpy(dtype=float)
        prox = distance_to_proximity(dist, scale)
        return dist, prox
    except Exception as e:
        print(f"Не удалось посчитать intersection proximity: {e}")
        return np.full(len(grid), np.nan, dtype=float), np.zeros(len(grid), dtype=float)


def smooth_on_regular_grid(arr: np.ndarray, valid_mask: np.ndarray, passes=2):
    work = arr.astype(float).copy()
    valid = valid_mask.astype(bool)

    for _ in range(int(passes)):
        num = np.zeros_like(work, dtype=float)
        den = np.zeros_like(work, dtype=float)
        finite_work = np.where(np.isfinite(work) & valid, work, 0.0)
        finite_mask = (np.isfinite(work) & valid).astype(float)

        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                shifted = np.roll(np.roll(finite_work, dr, axis=0), dc, axis=1)
                shifted_valid = np.roll(np.roll(finite_mask, dr, axis=0), dc, axis=1)

                if dr > 0:
                    shifted[:dr, :] = 0.0
                    shifted_valid[:dr, :] = 0.0
                elif dr < 0:
                    shifted[dr:, :] = 0.0
                    shifted_valid[dr:, :] = 0.0

                if dc > 0:
                    shifted[:, :dc] = 0.0
                    shifted_valid[:, :dc] = 0.0
                elif dc < 0:
                    shifted[:, dc:] = 0.0
                    shifted_valid[:, dc:] = 0.0

                num += shifted
                den += shifted_valid

        with np.errstate(divide="ignore", invalid="ignore"):
            smoothed = np.divide(num, den, out=np.full_like(num, np.nan), where=den > 0)
        work = np.where(valid, smoothed, np.nan)

    return work


def connected_component_filter_array(binary_arr, min_cells=2):
    binary_arr = np.asarray(binary_arr).astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, ncomp = label(binary_arr, structure=structure)
    keep = np.zeros_like(binary_arr, dtype=np.uint8)
    for cid in range(1, ncomp + 1):
        comp = (labeled == cid)
        if comp.sum() >= min_cells:
            keep[comp] = 1
    return keep.astype(np.uint8)


def save_gpkg_safe(gdf, path, layer=None):
    tmp = gdf.copy()
    # GeoPackage поддерживает только один активный geometry-столбец.
    for col in list(tmp.columns):
        if col != tmp.geometry.name and hasattr(tmp[col], "geom_type"):
            tmp = tmp.drop(columns=[col])
    tmp = tmp.drop(columns=["centroid"], errors="ignore")
    if layer:
        tmp.to_file(path, layer=layer, driver="GPKG")
    else:
        tmp.to_file(path, driver="GPKG")




## 2. Загрузка слоёв и построение сетки

In [ ]:



layers_raw = load_shapefiles(SHP_DIR)
print("Слои:", sorted(layers_raw.keys()))

for logical_name, layer_name in LAYER_MAP.items():
    if layer_name not in layers_raw:
        raise FileNotFoundError(f"Не найден обязательный слой {logical_name}: {layer_name}")

mask_name = LAYER_MAP["mask"]
target_crs = choose_target_crs(layers_raw[mask_name])
layers = harmonize_layers(layers_raw, target_crs)

print("Используемые слои:", LAYER_MAP)
point_candidates = discover_point_layers(layers, exclude_names=set(LAYER_MAP.values()))
print("Точечные слои-кандидаты:", point_candidates)

grid, GRID_SHAPE, GRID_BOUNDS = build_grid(layers[LAYER_MAP["mask"]], CELL_SIZE)
rows, cols = GRID_SHAPE
minx, miny, maxx, maxy = GRID_BOUNDS

print("Ячеек в сетке:", len(grid))
print("Размер сетки (rows, cols):", GRID_SHAPE)
print("Границы:", GRID_BOUNDS)




## 3. Расчёт признаков факторов

In [ ]:
#
# SOM не получает готовую формулу перспективности. Он получает набор признаков по ячейкам:
# - близость к факторам;
# - ранговые расстояния;
# - бинарные буферы `500/1000/1500 м`;
# - несколько простых логических комбинаций.
#
# Дальше SOM сама группирует похожие сочетания факторов.


# Базовые distance/proximity-признаки
for key in ["facies", "paleo", "struct", "magm", "tect1", "tect2"]:
    layer_name = LAYER_MAP[key]
    dist, prox = compute_distance_feature(grid, layers[layer_name], SCALES[key])
    grid[f"dist_{key}"] = dist
    grid[f"prox_{key}"] = prox

# Агрегированная тектоника: один общий tect_any вместо двух сильно дублирующих направлений.
grid["dist_tect_any"] = np.minimum(grid["dist_tect1"].to_numpy(float), grid["dist_tect2"].to_numpy(float))
grid["prox_tect_any"] = distance_to_proximity(grid["dist_tect_any"], scale=1000.0)

# Ранговое кодирование расстояний: 1 = ближе, 0 = дальше.
for key in ["facies", "paleo", "struct", "magm", "tect1", "tect2", "tect_any"]:
    s = pd.Series(grid[f"dist_{key}"].astype(float))
    grid[f"rank_near_{key}"] = 1.0 - s.rank(method="average", pct=True).to_numpy()
    grid[f"rank_near_{key}"] = grid[f"rank_near_{key}"].fillna(0.0)

# =========================
# NODE-AWARE FEATURE SET v3
# =========================
# Цель v3 — не просто подсветить разломные линии, а дать SOM признаки структурных узлов:
# 1) сколько разнородных факторов совпало в радиусе;
# 2) сколько парных/тройных сочетаний факторов активно;
# 3) близость к реальным геометрическим пересечениям слоёв.

NON_TECT_KEYS = ["facies", "paleo", "struct", "magm"]
ALL_KEYS = ["facies", "paleo", "struct", "magm", "tect1", "tect2"]

# Бинарные буферы считаем для всех факторов, но в признаки индивидуальные tect1/tect2 не подаём.
for key in ALL_KEYS:
    for b in BUFFER_DISTANCES:
        grid[f"buf_{key}_{b}"] = (grid[f"dist_{key}"] <= b).astype(int)

PAIR_DEFS = []
TRIPLE_DEFS = []

for b in BUFFER_DISTANCES:
    grid[f"tect_any_{b}"] = ((grid[f"buf_tect1_{b}"] == 1) | (grid[f"buf_tect2_{b}"] == 1)).astype(int)
    grid[f"tect_cross_{b}"] = ((grid[f"buf_tect1_{b}"] == 1) & (grid[f"buf_tect2_{b}"] == 1)).astype(int)

    # Активные классы факторов считаем по 5 классам: facies, paleo, struct, magm, tect_any.
    active_cols = [f"buf_{key}_{b}" for key in NON_TECT_KEYS] + [f"tect_any_{b}"]
    grid[f"active_factor_count_{b}"] = grid[active_cols].sum(axis=1)
    grid[f"multi_factor_3plus_{b}"] = (grid[f"active_factor_count_{b}"] >= 3).astype(int)
    grid[f"multi_factor_4plus_{b}"] = (grid[f"active_factor_count_{b}"] >= 4).astype(int)

    # Парные узлы: не все возможные пары, а геологически читаемые сочетания.
    pair_exprs = {
        f"node_tect_magm_{b}":      (grid[f"tect_any_{b}"] & grid[f"buf_magm_{b}"]),
        f"node_tect_struct_{b}":    (grid[f"tect_any_{b}"] & grid[f"buf_struct_{b}"]),
        f"node_tect_paleo_{b}":     (grid[f"tect_any_{b}"] & grid[f"buf_paleo_{b}"]),
        f"node_tect_facies_{b}":    (grid[f"tect_any_{b}"] & grid[f"buf_facies_{b}"]),
        f"node_struct_paleo_{b}":   (grid[f"buf_struct_{b}"] & grid[f"buf_paleo_{b}"]),
        f"node_magm_struct_{b}":    (grid[f"buf_magm_{b}"] & grid[f"buf_struct_{b}"]),
        f"node_magm_paleo_{b}":     (grid[f"buf_magm_{b}"] & grid[f"buf_paleo_{b}"]),
        f"node_facies_paleo_{b}":   (grid[f"buf_facies_{b}"] & grid[f"buf_paleo_{b}"]),
    }
    for col, val in pair_exprs.items():
        grid[col] = val.astype(int)
    pair_cols = list(pair_exprs.keys())
    PAIR_DEFS.extend(pair_cols)

    # Тройные узлы: именно они чаще всего похожи на локальные перспективные центры.
    triple_exprs = {
        f"triple_tect_magm_struct_{b}":   (grid[f"tect_any_{b}"] & grid[f"buf_magm_{b}"] & grid[f"buf_struct_{b}"]),
        f"triple_tect_paleo_struct_{b}":  (grid[f"tect_any_{b}"] & grid[f"buf_paleo_{b}"] & grid[f"buf_struct_{b}"]),
        f"triple_tect_paleo_facies_{b}":  (grid[f"tect_any_{b}"] & grid[f"buf_paleo_{b}"] & grid[f"buf_facies_{b}"]),
        f"triple_tect_magm_paleo_{b}":    (grid[f"tect_any_{b}"] & grid[f"buf_magm_{b}"] & grid[f"buf_paleo_{b}"]),
        f"triple_facies_paleo_struct_{b}":(grid[f"buf_facies_{b}"] & grid[f"buf_paleo_{b}"] & grid[f"buf_struct_{b}"]),
    }
    for col, val in triple_exprs.items():
        grid[col] = val.astype(int)
    triple_cols = list(triple_exprs.keys())
    TRIPLE_DEFS.extend(triple_cols)

    grid[f"node_pair_count_{b}"] = grid[pair_cols].sum(axis=1)
    grid[f"node_triple_count_{b}"] = grid[triple_cols].sum(axis=1)
    grid[f"node_complexity_{b}"] = (
        0.35 * normalize_01(grid[f"active_factor_count_{b}"]) +
        0.35 * normalize_01(grid[f"node_pair_count_{b}"]) +
        0.30 * normalize_01(grid[f"node_triple_count_{b}"])
    )

# Близость к реальным геометрическим пересечениям слоёв.
INTERSECTION_PAIRS = [
    ("tect1", "tect2", "tect1_tect2"),
    ("tect1", "magm", "tect1_magm"),
    ("tect2", "magm", "tect2_magm"),
    ("tect1", "paleo", "tect1_paleo"),
    ("tect2", "paleo", "tect2_paleo"),
    ("tect1", "struct", "tect1_struct"),
    ("tect2", "struct", "tect2_struct"),
    ("magm", "struct", "magm_struct"),
    ("struct", "paleo", "struct_paleo"),
]
nodeprox_cols = []
for a, b, name in INTERSECTION_PAIRS:
    dist, prox = compute_intersection_proximity(grid, layers[LAYER_MAP[a]], layers[LAYER_MAP[b]], scale=1200.0)
    grid[f"dist_node_{name}"] = dist
    grid[f"prox_node_{name}"] = prox
    nodeprox_cols.append(f"prox_node_{name}")

# Единый диагностический индекс узловости. Он НЕ заменяет SOM, а помогает контролировать top-zone.
node_strength_components = []
for b in BUFFER_DISTANCES:
    node_strength_components += [
        f"node_complexity_{b}",
        f"node_pair_count_{b}",
        f"node_triple_count_{b}",
        f"tect_cross_{b}",
    ]
node_strength_components += nodeprox_cols
node_strength_matrix = grid[node_strength_components].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy()
node_strength_scaled = np.column_stack([robust_normalize_01(node_strength_matrix[:, i]) for i in range(node_strength_matrix.shape[1])])
grid["node_strength_raw"] = node_strength_scaled.mean(axis=1)
grid["node_strength"] = robust_normalize_01(grid["node_strength_raw"], 0.02, 0.98)

FEATURE_COLS = []

# Плавный фон: non-tect факторы + агрегированная тектоника.
FEATURE_COLS += [f"prox_{k}" for k in NON_TECT_KEYS]
FEATURE_COLS += ["prox_tect_any"]
FEATURE_COLS += [f"rank_near_{k}" for k in NON_TECT_KEYS]
FEATURE_COLS += ["rank_near_tect_any"]

# Буферы: индивидуально non-tect, тектоника только как aggregate.
for b in BUFFER_DISTANCES:
    FEATURE_COLS += [f"buf_{k}_{b}" for k in NON_TECT_KEYS]
    FEATURE_COLS += [
        f"tect_any_{b}",
        f"tect_cross_{b}",
        f"active_factor_count_{b}",
        f"multi_factor_3plus_{b}",
        f"multi_factor_4plus_{b}",
        f"node_pair_count_{b}",
        f"node_triple_count_{b}",
        f"node_complexity_{b}",
    ]

# Узловые пары/тройки и реальные геометрические intersection proximity.
FEATURE_COLS += PAIR_DEFS
FEATURE_COLS += TRIPLE_DEFS
FEATURE_COLS += nodeprox_cols
FEATURE_COLS += ["node_strength"]

# Удаляем возможные дубли.
FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS))

X_raw = grid[FEATURE_COLS].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy()
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_raw)
# После RobustScaler приведём каждый столбец к 0..1, чтобы SOM не доминировалась выбросами.
X_base = np.column_stack([robust_normalize_01(X_scaled[:, i]) for i in range(X_scaled.shape[1])]).astype(np.float32)

def feature_group_weight(feature_name: str) -> float:
    """
    v4: ручная балансировка групп признаков перед SOM.
    Это не готовый score перспективности, а изменение геометрии пространства признаков,
    чтобы SOM сильнее разделяла узлы наложения факторов, а не только одиночные линии разломов.
    """
    name = str(feature_name)
    node_markers = [
        "node_", "prox_node", "dist_node", "node_strength", "node_pair", "node_triple",
        "node_complexity", "multi_factor", "active_factor_count", "triple_", "tect_cross"
    ]
    if any(m in name for m in node_markers):
        return NODE_FEATURE_MULTIPLIER
    if "tect" in name:
        return TECT_FEATURE_MULTIPLIER
    if "struct" in name:
        return STRUCT_FEATURE_MULTIPLIER
    if "paleo" in name:
        return PALEO_FEATURE_MULTIPLIER
    if "magm" in name:
        return MAGM_FEATURE_MULTIPLIER
    if "facies" in name:
        return FACIES_FEATURE_MULTIPLIER
    return 1.0

FEATURE_WEIGHTS = np.array([feature_group_weight(f) for f in FEATURE_COLS], dtype=np.float32)
X = np.clip(X_base * FEATURE_WEIGHTS.reshape(1, -1), 0.0, FEATURE_CLIP_MAX).astype(np.float32)

feature_weight_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "som_input_weight": FEATURE_WEIGHTS,
}).sort_values("som_input_weight", ascending=False)

print("Количество признаков для SOM v4 node-weighted:", len(FEATURE_COLS))
print("Размер X:", X.shape)
print("Средняя node_strength:", float(grid["node_strength"].mean()))
print("Групповые веса признаков:")
display(feature_weight_df.head(80))




## 4. Evidence cells: известные проявления и геохимические признаки

In [ ]:
#
# Эти данные **не участвуют в обучении SOM**. Они используются после обучения, чтобы понять, какие SOM-нейроны чаще совпадают с известными проявлениями.



def collect_evidence_points(layers, point_names, candidates, target_crs):
    if point_names is None:
        names = [x["layer"] for x in candidates]
    else:
        names = [n for n in point_names if n in layers]

    point_gdfs = []
    for name in names:
        gdf = layers[name].copy()
        if len(gdf) == 0:
            continue
        try:
            gdf = to_crs_safe(gdf, target_crs)
        except Exception:
            pass
        gdf = repair_geometries(gdf)
        gdf["source_layer"] = name
        point_gdfs.append(gdf[["source_layer", "geometry"]].copy())

    if not point_gdfs:
        return gpd.GeoDataFrame({"source_layer": []}, geometry=[], crs=target_crs), []

    pts = pd.concat(point_gdfs, ignore_index=True)
    pts = gpd.GeoDataFrame(pts, geometry="geometry", crs=target_crs)
    return pts, names


def mark_evidence_cells(grid, layers, point_names, polygon_names, candidates, target_crs):
    grid_e = grid[["cell_id", "geometry"]].copy()
    evidence_cell_ids = set()

    points_raw, used_points = collect_evidence_points(layers, point_names, candidates, target_crs)
    points_plot = gpd.GeoDataFrame({"source_layer": []}, geometry=[], crs=target_crs)

    if len(points_raw) > 0:
        # intersects безопаснее, чем within, потому что точки на границе ячеек тоже попадут.
        sj = gpd.sjoin(points_raw, grid_e, how="inner", predicate="intersects")
        if len(sj) > 0:
            evidence_cell_ids.update(sj["cell_id"].astype(int).tolist())
            points_plot = sj.drop(columns=["index_right"], errors="ignore").copy()

    used_polygons = []
    if USE_EVIDENCE_POLYGONS:
        for name in polygon_names:
            if name not in layers:
                continue
            gdf = layers[name].copy()
            if len(gdf) == 0:
                continue
            geom_types = set(gdf.geom_type.dropna().unique().tolist())
            if geom_types.issubset({"Point", "MultiPoint"}):
                continue
            try:
                gdf = to_crs_safe(gdf, target_crs)
            except Exception:
                pass
            gdf = repair_geometries(gdf)
            if len(gdf) == 0:
                continue
            used_polygons.append(name)
            sj_poly = gpd.sjoin(grid_e, gdf[["geometry"]], how="inner", predicate="intersects")
            if len(sj_poly) > 0:
                evidence_cell_ids.update(sj_poly["cell_id"].astype(int).tolist())

    evidence = np.zeros(len(grid), dtype=np.uint8)
    if evidence_cell_ids:
        evidence[list(evidence_cell_ids)] = 1
    return evidence, points_raw, points_plot, used_points, used_polygons


grid["evidence_cell"], evidence_points_raw, evidence_points_plot, used_point_layers, used_polygon_layers = mark_evidence_cells(
    grid=grid,
    layers=layers,
    point_names=EVIDENCE_POINT_LAYER_NAMES,
    polygon_names=EVIDENCE_POLYGON_LAYER_NAMES,
    candidates=point_candidates,
    target_crs=target_crs,
)

print("Используем point-слои evidence:", used_point_layers)
print("Используем polygon-слои evidence:", used_polygon_layers)
print("Количество evidence points raw:", len(evidence_points_raw))
print("Количество evidence points внутри сетки:", len(evidence_points_plot))
print("Количество evidence cells:", int(grid["evidence_cell"].sum()))
print("Доля evidence cells:", float(grid["evidence_cell"].mean()))




## 5. Реализация SOM без внешней библиотеки MiniSom

In [ ]:
#
# Чтобы не было ошибки `ModuleNotFoundError: No module named 'minisom'`, здесь SOM реализован напрямую на `NumPy`.



class NumpySOM:
    def __init__(self, x_dim, y_dim, input_dim, seed=42):
        self.x_dim = int(x_dim)
        self.y_dim = int(y_dim)
        self.input_dim = int(input_dim)
        self.rng = np.random.default_rng(seed)
        self.weights = None
        gx, gy = np.meshgrid(np.arange(self.x_dim), np.arange(self.y_dim), indexing="ij")
        self.grid_x = gx
        self.grid_y = gy

    def initialize(self, X):
        X = np.asarray(X, dtype=np.float32)
        n = len(X)
        if n == 0:
            raise ValueError("Пустая матрица X для SOM.")
        idx = self.rng.choice(n, size=self.x_dim * self.y_dim, replace=True)
        self.weights = X[idx].reshape(self.x_dim, self.y_dim, self.input_dim).astype(np.float32)
        self.weights += self.rng.normal(0, 0.01, size=self.weights.shape).astype(np.float32)

    def _bmu_single(self, x):
        diff = self.weights - x.reshape(1, 1, -1)
        d2 = np.sum(diff * diff, axis=2)
        flat_idx = int(np.argmin(d2))
        bx, by = np.unravel_index(flat_idx, (self.x_dim, self.y_dim))
        return bx, by, float(np.sqrt(d2[bx, by]))

    def train(self, X, epochs=40, max_samples_per_epoch=5000,
              start_lr=0.45, end_lr=0.05, start_sigma=6.0, end_sigma=1.0,
              print_every=5):
        X = np.asarray(X, dtype=np.float32)
        if self.weights is None:
            self.initialize(X)
        n = len(X)
        history = []

        for ep in range(1, epochs + 1):
            t = (ep - 1) / max(1, epochs - 1)
            lr = start_lr * ((end_lr / start_lr) ** t)
            sigma = start_sigma * ((end_sigma / start_sigma) ** t)
            sigma2 = max(sigma * sigma, 1e-6)

            sample_n = min(int(max_samples_per_epoch), n)
            idxs = self.rng.choice(n, size=sample_n, replace=False)
            self.rng.shuffle(idxs)

            qerr_sum = 0.0
            for idx in idxs:
                x = X[idx]
                bx, by, qerr = self._bmu_single(x)
                qerr_sum += qerr

                gd2 = (self.grid_x - bx) ** 2 + (self.grid_y - by) ** 2
                h = np.exp(-gd2 / (2.0 * sigma2)).astype(np.float32)
                self.weights += lr * h[:, :, None] * (x.reshape(1, 1, -1) - self.weights)

            qerr_mean = qerr_sum / sample_n
            history.append({"epoch": ep, "lr": float(lr), "sigma": float(sigma), "quantization_error": float(qerr_mean)})
            if ep == 1 or ep % print_every == 0 or ep == epochs:
                print(f"Epoch {ep:03d} | q_error={qerr_mean:.6f} | lr={lr:.4f} | sigma={sigma:.3f}")
        return pd.DataFrame(history)

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        bmu_x = np.zeros(len(X), dtype=np.int32)
        bmu_y = np.zeros(len(X), dtype=np.int32)
        qerr = np.zeros(len(X), dtype=np.float32)
        for i, x in enumerate(X):
            bx, by, qe = self._bmu_single(x)
            bmu_x[i] = bx
            bmu_y[i] = by
            qerr[i] = qe
        return bmu_x, bmu_y, qerr




## 6. Обучение SOM

In [ ]:



som = NumpySOM(SOM_X, SOM_Y, X.shape[1], seed=SEED)
train_history = som.train(
    X,
    epochs=SOM_EPOCHS,
    max_samples_per_epoch=SOM_MAX_SAMPLES_PER_EPOCH,
    start_lr=SOM_START_LR,
    end_lr=SOM_END_LR,
    start_sigma=SOM_START_SIGMA,
    end_sigma=SOM_END_SIGMA,
    print_every=SOM_PRINT_EVERY,
)

bmu_x, bmu_y, qerr = som.predict(X)
grid["som_x"] = bmu_x
grid["som_y"] = bmu_y
grid["som_id"] = grid["som_x"] * SOM_Y + grid["som_y"]
grid["som_qerr"] = qerr
print("SOM обучена.")
print("Средняя quantization error:", float(np.mean(qerr)))




## 7. Интерпретация SOM-нейронов по evidence cells

In [ ]:
#
# SOM обучалась без меток. Теперь мы смотрим, какие SOM-нейроны чаще совпадают с evidence cells. Чтобы 68–100 ячеек не давали слишком шумный результат, используется сглаживание по SOM-сетке и байесовский prior.



neuron_cell_count = np.zeros((SOM_X, SOM_Y), dtype=float)
neuron_evidence_count = np.zeros((SOM_X, SOM_Y), dtype=float)

for bx, by, ev in zip(grid["som_x"].to_numpy(), grid["som_y"].to_numpy(), grid["evidence_cell"].to_numpy()):
    neuron_cell_count[int(bx), int(by)] += 1.0
    neuron_evidence_count[int(bx), int(by)] += float(ev)

base_rate = float(grid["evidence_cell"].mean())
print("Base evidence rate:", base_rate)

smooth_evidence = gaussian_filter(neuron_evidence_count, sigma=SOM_SCORE_SMOOTH_SIGMA, mode="nearest")
smooth_cells = gaussian_filter(neuron_cell_count, sigma=SOM_SCORE_SMOOTH_SIGMA, mode="nearest")

posterior_rate = (smooth_evidence + EVIDENCE_PRIOR_STRENGTH * base_rate) / (smooth_cells + EVIDENCE_PRIOR_STRENGTH + 1e-9)
lift = posterior_rate / (base_rate + 1e-9)
log_lift = np.log1p(np.maximum(lift, 0.0))
evidence_score = robust_normalize_01(log_lift, 0.02, 0.98)

# v4: SOM-нейроны оцениваются не только по evidence, но и по средним узловым свойствам.
# Это нужно, чтобы фон карты тоже реагировал на узлы пересечения факторов, а не только top-zone.
node_score_neuron = np.zeros((SOM_X, SOM_Y), dtype=float)
active_score_neuron = np.zeros((SOM_X, SOM_Y), dtype=float)

for bx in range(SOM_X):
    for by in range(SOM_Y):
        mask = (grid["som_x"] == bx) & (grid["som_y"] == by)
        if mask.any():
            node_score_neuron[bx, by] = float(grid.loc[mask, "node_strength"].mean())
            active_score_neuron[bx, by] = float(
                0.5 * grid.loc[mask, "active_factor_count_1000"].mean() +
                0.5 * grid.loc[mask, "active_factor_count_1500"].mean()
            )

node_score_neuron = robust_normalize_01(node_score_neuron, 0.02, 0.98)
active_score_neuron = robust_normalize_01(active_score_neuron, 0.02, 0.98)

if grid["evidence_cell"].sum() > 0 and np.nanmax(evidence_score) > 0:
    neuron_score = (
        NEURON_EVIDENCE_WEIGHT * evidence_score +
        NEURON_NODE_WEIGHT * node_score_neuron +
        NEURON_ACTIVE_WEIGHT * active_score_neuron
    )
else:
    print("Evidence cells не найдены или evidence-score пустой. Используется node-aware fallback.")
    neuron_score = (
        0.65 * node_score_neuron +
        0.35 * active_score_neuron
    )

neuron_score = robust_normalize_01(neuron_score, 0.02, 0.98)

# Небольшая поправка на близость ячейки к своему BMU: это не новый метод, а SOM-уверенность.
qerr_similarity = 1.0 - robust_normalize_01(grid["som_qerr"].to_numpy(), 0.02, 0.98)

som_score_per_cell = np.array([neuron_score[int(x), int(y)] for x, y in zip(grid["som_x"], grid["som_y"])], dtype=float)
grid["som_cluster_score"] = som_score_per_cell
grid["som_similarity"] = qerr_similarity
grid["prospectivity_raw"] = (
    (1.0 - INTRA_NEURON_WEIGHT - NODE_SCORE_WEIGHT) * grid["som_cluster_score"] +
    INTRA_NEURON_WEIGHT * grid["som_similarity"] +
    NODE_SCORE_WEIGHT * grid["node_strength"]
)
grid["prospectivity_raw"] = robust_normalize_01(grid["prospectivity_raw"].to_numpy(), 0.02, 0.98)

# Сводка по SOM-нейронам
summary_rows = []
for bx in range(SOM_X):
    for by in range(SOM_Y):
        mask = (grid["som_x"] == bx) & (grid["som_y"] == by)
        row = {
            "som_x": bx,
            "som_y": by,
            "som_id": bx * SOM_Y + by,
            "cell_count": int(mask.sum()),
            "evidence_count": int(grid.loc[mask, "evidence_cell"].sum()) if mask.any() else 0,
            "raw_evidence_rate": float(grid.loc[mask, "evidence_cell"].mean()) if mask.any() else np.nan,
            "posterior_rate_smoothed": float(posterior_rate[bx, by]),
            "lift_smoothed": float(lift[bx, by]),
            "evidence_score": float(evidence_score[bx, by]),
            "node_score_neuron": float(node_score_neuron[bx, by]),
            "active_score_neuron": float(active_score_neuron[bx, by]),
            "neuron_score": float(neuron_score[bx, by]),
        }
        for f in ["active_factor_count_1000", "active_factor_count_1500", "node_strength", "node_pair_count_1500", "node_triple_count_1500", "tect_cross_1500", "node_tect_magm_1500", "node_tect_struct_1500", "triple_tect_paleo_struct_1500"]:
            row[f"mean_{f}"] = float(grid.loc[mask, f].mean()) if mask.any() else np.nan
        summary_rows.append(row)

som_summary = pd.DataFrame(summary_rows).sort_values("neuron_score", ascending=False)
som_summary.head(20)




## 8. Пространственное сглаживание, `prognoz`, `top_zone`

In [ ]:


valid_mask = np.zeros((rows, cols), dtype=bool)
prospectivity_arr = np.full((rows, cols), np.nan, dtype=float)

for _, row in grid.iterrows():
    r = int(row["row"])
    c = int(row["col"])
    valid_mask[r, c] = True
    prospectivity_arr[r, c] = float(row["prospectivity_raw"])

prospectivity_arr = smooth_on_regular_grid(prospectivity_arr, valid_mask, passes=SMOOTH_PASSES)
prospectivity_arr = robust_normalize_01(prospectivity_arr, 0.02, 0.98)
prospectivity_arr = np.where(valid_mask, prospectivity_arr, np.nan)
prognoz_arr = 1.0 - prospectivity_arr

node_strength_arr = np.full((rows, cols), np.nan, dtype=float)
for _, row in grid.iterrows():
    r = int(row["row"])
    c = int(row["col"])
    node_strength_arr[r, c] = float(row["node_strength"])
node_strength_arr = smooth_on_regular_grid(node_strength_arr, valid_mask, passes=1)
node_strength_arr = robust_normalize_01(node_strength_arr, 0.02, 0.98)
node_strength_arr = np.where(valid_mask, node_strength_arr, np.nan)


def build_top_zone_arr(prospectivity_arr, valid_mask):
    """
    Строгий top-zone для v3.
    Идея: не заливать всю перспективную полосу, а оставить компактные локальные максимумы.
    Если доля top_zone слишком большая, порог автоматически повышается.
    """
    q_candidates = [TOP_ZONE_Q, 0.996, 0.9975, 0.999]
    q_candidates = sorted(set([float(q) for q in q_candidates if 0.0 < q < 1.0]))
    last_top = None
    last_q = None
    last_thr = None

    for q in q_candidates:
        thr = float(np.nanquantile(prospectivity_arr[valid_mask], q))
        local_max = maximum_filter(np.nan_to_num(prospectivity_arr, nan=0.0), size=LOCAL_MAX_SIZE)
        node_thr = float(np.nanquantile(node_strength_arr[valid_mask], NODE_TOP_ZONE_Q))
        peak_mask = (
            (prospectivity_arr >= thr) &
            (prospectivity_arr >= local_max - 1e-9) &
            (node_strength_arr >= node_thr) &
            valid_mask
        )
        top = connected_component_filter_array(peak_mask.astype(np.uint8), min_cells=MIN_TOP_COMPONENT_CELLS)
        share = float(top.sum() / max(1, valid_mask.sum()))
        last_top, last_q, last_thr = top, q, thr
        if top.sum() > 0 and share <= MAX_TOP_ZONE_SHARE:
            return top, q, thr, share

    # fallback: если локальные максимумы ничего не оставили, берём очень строгий квантиль без local-max
    if last_top is None or last_top.sum() == 0:
        q = max(TOP_ZONE_Q, 0.997)
        thr = float(np.nanquantile(prospectivity_arr[valid_mask], q))
        node_thr = float(np.nanquantile(node_strength_arr[valid_mask], NODE_TOP_ZONE_Q))
        top = ((prospectivity_arr >= thr) & (node_strength_arr >= node_thr) & valid_mask).astype(np.uint8)
        top = connected_component_filter_array(top, min_cells=MIN_TOP_COMPONENT_CELLS)
        share = float(top.sum() / max(1, valid_mask.sum()))
        return top, q, thr, share

    return last_top, last_q, last_thr, float(last_top.sum() / max(1, valid_mask.sum()))


top_zone_arr, effective_top_zone_q, q_top, top_zone_share = build_top_zone_arr(prospectivity_arr, valid_mask)

# Записываем обратно в grid
grid["prospectivity"] = np.nan
grid["prognoz"] = np.nan
grid["top_zone"] = 0

for idx, row in grid[["row", "col"]].iterrows():
    r = int(row["row"])
    c = int(row["col"])
    grid.at[idx, "prospectivity"] = float(prospectivity_arr[r, c])
    grid.at[idx, "prognoz"] = float(prognoz_arr[r, c])
    grid.at[idx, "top_zone"] = int(top_zone_arr[r, c])

print("Эффективный квантиль top-zone:", effective_top_zone_q)
print("Порог top-zone:", q_top)
print("Ячеек top_zone:", int(grid["top_zone"].sum()))
print("Доля top_zone:", float(grid["top_zone"].mean()))




## 9. Метрики проверки по evidence cells

In [ ]:
#
# Это не supervised train/test. Это диагностическая проверка: попадают ли известные проявления и геохимические признаки в верхние процентили карты.


def hitrate_at_quantile(grid, score_col, evidence_col, q):
    threshold = float(grid[score_col].quantile(q))
    top = grid[score_col] >= threshold
    ev = grid[evidence_col].astype(bool)
    n_ev = int(ev.sum())
    if n_ev == 0:
        return {"q": q, "threshold": threshold, "evidence_in_top": None, "hitrate": None, "top_area_share": float(top.mean())}
    return {
        "q": q,
        "threshold": threshold,
        "evidence_in_top": int((top & ev).sum()),
        "hitrate": float((top & ev).sum() / n_ev),
        "top_area_share": float(top.mean()),
        "lift": float(((top & ev).sum() / max(1, top.sum())) / max(1e-9, ev.mean())) if top.sum() > 0 else None,
    }


def safe_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 3:
        return None
    if np.nanstd(a[mask]) <= 1e-12 or np.nanstd(b[mask]) <= 1e-12:
        return None
    return float(np.corrcoef(a[mask], b[mask])[0, 1])

# Диагностика: не стала ли карта просто картой тектоники.
key_diagnostic_features = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm", "prox_tect_any",
    "active_factor_count_1000", "active_factor_count_1500",
    "node_strength", "node_pair_count_1500", "node_triple_count_1500",
    "tect_any_1500", "tect_cross_1500",
    "node_tect_magm_1500", "node_tect_struct_1500", "node_tect_paleo_1500",
    "triple_tect_magm_struct_1500", "triple_tect_paleo_struct_1500", "triple_tect_magm_paleo_1500",
    "prox_node_tect1_tect2", "prox_node_tect1_magm", "prox_node_tect2_magm",
    "prox_node_tect1_struct", "prox_node_tect2_struct", "prox_node_struct_paleo"
]
feature_corrs = []
for f in key_diagnostic_features:
    if f in grid.columns:
        feature_corrs.append({"feature": f, "corr_with_prospectivity": safe_corr(grid[f], grid["prospectivity"])})
feature_corrs_df = pd.DataFrame(feature_corrs).sort_values("corr_with_prospectivity", key=lambda s: s.abs(), ascending=False)
display(feature_corrs_df)

metrics = {
    "method": "SOM_unsupervised_ML_v4_nodeweighted",
    "cell_size": CELL_SIZE,
    "grid_cells": int(len(grid)),
    "grid_shape": [int(rows), int(cols)],
    "feature_count": int(len(FEATURE_COLS)),
    "node_score_weight": float(NODE_SCORE_WEIGHT),
    "neuron_evidence_weight": float(NEURON_EVIDENCE_WEIGHT),
    "neuron_node_weight": float(NEURON_NODE_WEIGHT),
    "neuron_active_weight": float(NEURON_ACTIVE_WEIGHT),
    "tect_feature_multiplier": float(TECT_FEATURE_MULTIPLIER),
    "facies_feature_multiplier": float(FACIES_FEATURE_MULTIPLIER),
    "magm_feature_multiplier": float(MAGM_FEATURE_MULTIPLIER),
    "paleo_feature_multiplier": float(PALEO_FEATURE_MULTIPLIER),
    "struct_feature_multiplier": float(STRUCT_FEATURE_MULTIPLIER),
    "node_feature_multiplier": float(NODE_FEATURE_MULTIPLIER),
    "node_top_zone_q": float(NODE_TOP_ZONE_Q),
    "mean_node_strength_all": float(grid["node_strength"].mean()),
    "mean_node_strength_evidence": float(grid.loc[grid["evidence_cell"] == 1, "node_strength"].mean()) if grid["evidence_cell"].sum() > 0 else None,
    "som_x": int(SOM_X),
    "som_y": int(SOM_Y),
    "som_epochs": int(SOM_EPOCHS),
    "evidence_point_layers": used_point_layers,
    "evidence_polygon_layers": used_polygon_layers,
    "use_evidence_polygons": bool(USE_EVIDENCE_POLYGONS),
    "evidence_points_raw": int(len(evidence_points_raw)),
    "evidence_points_inside_grid": int(len(evidence_points_plot)),
    "evidence_cells": int(grid["evidence_cell"].sum()),
    "evidence_cell_share": float(grid["evidence_cell"].mean()),
    "effective_top_zone_q": float(effective_top_zone_q),
    "top_zone_threshold": float(q_top),
    "top_zone_cells": int(grid["top_zone"].sum()),
    "top_zone_share": float(grid["top_zone"].mean()),
    "mean_prospectivity_all": float(grid["prospectivity"].mean()),
    "mean_prospectivity_evidence": float(grid.loc[grid["evidence_cell"] == 1, "prospectivity"].mean()) if grid["evidence_cell"].sum() > 0 else None,
    "hitrate_top_5pct": hitrate_at_quantile(grid, "prospectivity", "evidence_cell", 0.95),
    "hitrate_top_10pct": hitrate_at_quantile(grid, "prospectivity", "evidence_cell", 0.90),
    "hitrate_top_15pct": hitrate_at_quantile(grid, "prospectivity", "evidence_cell", 0.85),
    "hitrate_top_20pct": hitrate_at_quantile(grid, "prospectivity", "evidence_cell", 0.80),
    "feature_correlations_with_prospectivity": feature_corrs,
}

print(json.dumps(metrics, ensure_ascii=False, indent=2))




## 10. Визуализация

In [ ]:
#
# Левая карта: `prospectivity`, где больше = перспективнее. Правая карта: `prognoz = 1 - prospectivity`, где меньше = перспективнее.



def plot_result():
    fig, axes = plt.subplots(1, 2, figsize=(18, 9), dpi=140)
    extent = (minx, maxx, miny, maxy)

    im0 = axes[0].imshow(prospectivity_arr, origin="lower", extent=extent, cmap="RdYlBu_r", vmin=0, vmax=1)
    axes[0].set_title("SOM v4 node-weighted prospectivity")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(prognoz_arr, origin="lower", extent=extent, cmap="RdYlBu", vmin=0, vmax=1)
    axes[1].set_title("SOM v4: Prognoz = 1 - prospectivity")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    for ax in axes:
        # Маска и несколько основных контуров
        try:
            layers[LAYER_MAP["mask"]].boundary.plot(ax=ax, color="black", linewidth=0.5, alpha=0.7)
        except Exception:
            pass
        for key, color, lw, alpha in [
            ("tect1", "black", 0.35, 0.5),
            ("tect2", "black", 0.35, 0.5),
            ("magm", "dimgray", 0.5, 0.45),
            ("paleo", "gray", 0.35, 0.35),
            ("struct", "gray", 0.35, 0.35),
        ]:
            try:
                gdf = layers[LAYER_MAP[key]]
                if len(gdf) > 0:
                    gdf.boundary.plot(ax=ax, color=color, linewidth=lw, alpha=alpha)
            except Exception:
                pass

        top_gdf = grid[grid["top_zone"] == 1].copy()
        if len(top_gdf) > 0:
            top_gdf.plot(ax=ax, color="yellow", edgecolor="black", linewidth=0.35, alpha=0.95, label="Top zone")

        if SHOW_POINTS and len(evidence_points_plot) > 0:
            evidence_points_plot.plot(ax=ax, color="yellow", edgecolor="black", markersize=10, alpha=0.95, zorder=5)

        ax.set_xlim(minx, maxx)
        ax.set_ylim(miny, maxy)
        ax.set_aspect("equal")
        ax.axis("off")

    plt.tight_layout()
    out_png = OUT_DIR / "som_v4_nodeweighted_result.png"
    plt.savefig(out_png, bbox_inches="tight", dpi=220)
    plt.show()
    print("Сохранено:", out_png)

plot_result()




## 11. Сохранение результатов

In [ ]:



# Сохраняем таблицы
som_summary.to_csv(OUT_DIR / "som_neuron_summary.csv", index=False, encoding="utf-8-sig")
feature_corrs_df.to_csv(OUT_DIR / "feature_correlations_with_prospectivity.csv", index=False, encoding="utf-8-sig")
feature_weight_df.to_csv(OUT_DIR / "som_feature_weights.csv", index=False, encoding="utf-8-sig")
train_history.to_csv(OUT_DIR / "som_train_history.csv", index=False, encoding="utf-8-sig")

# CSV по ячейкам без geometry
csv_df = grid.drop(columns=["geometry", "centroid"], errors="ignore").copy()
csv_df.to_csv(OUT_DIR / "som_grid_attributes.csv", index=False, encoding="utf-8-sig")

# Метрики и параметры
with open(OUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

params = {
    "BASE_DIR": str(BASE_DIR),
    "SHP_DIR": str(SHP_DIR),
    "OUT_DIR": str(OUT_DIR),
    "CELL_SIZE": CELL_SIZE,
    "LAYER_MAP": LAYER_MAP,
    "FEATURE_COLS": FEATURE_COLS,
    "FEATURE_WEIGHTS": {str(f): float(w) for f, w in zip(FEATURE_COLS, FEATURE_WEIGHTS)},
    "SOM_X": SOM_X,
    "SOM_Y": SOM_Y,
    "SOM_EPOCHS": SOM_EPOCHS,
    "SOM_MAX_SAMPLES_PER_EPOCH": SOM_MAX_SAMPLES_PER_EPOCH,
    "SOM_START_LR": SOM_START_LR,
    "SOM_END_LR": SOM_END_LR,
    "SOM_START_SIGMA": SOM_START_SIGMA,
    "SOM_END_SIGMA": SOM_END_SIGMA,
    "SOM_SCORE_SMOOTH_SIGMA": SOM_SCORE_SMOOTH_SIGMA,
    "EVIDENCE_PRIOR_STRENGTH": EVIDENCE_PRIOR_STRENGTH,
    "INTRA_NEURON_WEIGHT": INTRA_NEURON_WEIGHT,
    "NODE_SCORE_WEIGHT": NODE_SCORE_WEIGHT,
    "NEURON_EVIDENCE_WEIGHT": NEURON_EVIDENCE_WEIGHT,
    "NEURON_NODE_WEIGHT": NEURON_NODE_WEIGHT,
    "NEURON_ACTIVE_WEIGHT": NEURON_ACTIVE_WEIGHT,
    "TECT_FEATURE_MULTIPLIER": TECT_FEATURE_MULTIPLIER,
    "FACIES_FEATURE_MULTIPLIER": FACIES_FEATURE_MULTIPLIER,
    "MAGM_FEATURE_MULTIPLIER": MAGM_FEATURE_MULTIPLIER,
    "PALEO_FEATURE_MULTIPLIER": PALEO_FEATURE_MULTIPLIER,
    "STRUCT_FEATURE_MULTIPLIER": STRUCT_FEATURE_MULTIPLIER,
    "NODE_FEATURE_MULTIPLIER": NODE_FEATURE_MULTIPLIER,
    "FEATURE_CLIP_MAX": FEATURE_CLIP_MAX,
    "NODE_TOP_ZONE_Q": NODE_TOP_ZONE_Q,
    "SMOOTH_PASSES": SMOOTH_PASSES,
    "TOP_ZONE_Q": TOP_ZONE_Q,
    "effective_top_zone_q": float(effective_top_zone_q),
    "MAX_TOP_ZONE_SHARE": MAX_TOP_ZONE_SHARE,
    "LOCAL_MAX_SIZE": LOCAL_MAX_SIZE,
    "MIN_TOP_COMPONENT_CELLS": MIN_TOP_COMPONENT_CELLS,
    "BUFFER_DISTANCES": BUFFER_DISTANCES,
    "USE_EVIDENCE_POLYGONS": USE_EVIDENCE_POLYGONS,
    "SCALES": SCALES,
}
with open(OUT_DIR / "run_params.json", "w", encoding="utf-8") as f:
    json.dump(params, f, ensure_ascii=False, indent=2)

# SOM weights
np.save(OUT_DIR / "som_weights.npy", som.weights)
np.save(OUT_DIR / "neuron_score.npy", neuron_score)

# GeoPackage
if SAVE_FULL_GRID_GPKG:
    save_gpkg_safe(grid, OUT_DIR / "som_result_cells.gpkg")

top_zone = grid[grid["top_zone"] == 1].copy()
if len(top_zone) > 0:
    save_gpkg_safe(top_zone, OUT_DIR / "top_zone_cells.gpkg")

if len(evidence_points_raw) > 0:
    save_gpkg_safe(evidence_points_raw, OUT_DIR / "evidence_points_raw.gpkg")
if len(evidence_points_plot) > 0:
    save_gpkg_safe(evidence_points_plot, OUT_DIR / "evidence_points_inside_grid.gpkg")

print("Готово. Результаты сохранены в:", OUT_DIR)
print("Основные файлы:")
for p in [
    "som_v4_nodeweighted_result.png",
    "som_feature_weights.csv",
    "feature_correlations_with_prospectivity.csv",
    "som_result_cells.gpkg",
    "top_zone_cells.gpkg",
    "som_grid_attributes.csv",
    "som_neuron_summary.csv",
    "metrics.json",
    "run_params.json",
]:
    print(" -", OUT_DIR / p)




## Как интерпретировать результат v4

In [ ]:
#
# - `prospectivity` — итоговая перспективность SOM-подхода, где `1` ближе к SOM-зонам, чаще связанным с evidence cells.
# - `prognoz = 1 - prospectivity` — обратная шкала, где меньшие значения лучше.
# - `som_id`, `som_x`, `som_y` — SOM-нейрон, к которому отнесена ячейка.
# - `som_cluster_score` — score SOM-нейрона по совпадению с evidence cells.
# - `som_qerr` — ошибка квантования: насколько ячейка хорошо представлена своим SOM-нейроном.
# - `top_zone` — компактные зоны из верхней части прогноза.
#
# Что улучшено в v4:
#
# - одиночная тектоника ослаблена через `TECT_FEATURE_MULTIPLIER`;
# - узловые признаки усилены через `NODE_FEATURE_MULTIPLIER`;
# - SOM-нейроны оцениваются по evidence, node_strength и числу активных факторов;
# - `top_zone` стала компактнее и сильнее привязана к узлам пересечения;
# - добавлены диагностики `som_feature_weights.csv` и `feature_correlations_with_prospectivity.csv`.
#
# Если карта всё ещё слишком повторяет разломы:
#
# - уменьшите `INTRA_NEURON_WEIGHT` до `0.03`;
# - поставьте `USE_EVIDENCE_POLYGONS = False`, если было `True`;
# - уберите из `FEATURE_COLS` `tect_cross_1000` и `tect_cross_1500`.
#
# Если top-зона слишком большая:
#
# - увеличьте `TOP_ZONE_Q` до `0.997`;
# - уменьшите `MAX_TOP_ZONE_SHARE` до `0.006`.
#
# Если top-зона слишком маленькая:
#
# - уменьшите `TOP_ZONE_Q` до `0.99`;
# - уменьшите `MIN_TOP_COMPONENT_CELLS` до `2`.

